---
## 📝 Tarea — Clasificación supervisada de un glaciar peruano con SVM

Aplica el **mismo flujo de trabajo** de la sección 7, pero ahora sobre un **glaciar peruano**, usando SVM (en lugar de K-Means no supervisado de la tarea anterior).

**Glaciar sugerido:** Coropuna (Arequipa) — el mismo que usaste en la tarea de K-Means, para poder comparar ambos enfoques sobre la misma área. Puedes usar otro glaciar si lo prefieres (Pastoruri en Áncash, Quelccaya en Cusco, algún nevado de la Cordillera Vilcanota, etc.) — solo ajusta las coordenadas.

### Clases sugeridas para el glaciar

| Código | Clase |
|--------|-------|
| 0 | Agua (lagunas proglaciares) |
| 1 | Nieve / Hielo |
| 2 | Roca expuesta / morrena |
| 3 | Pastizal ralo (zonas bajas) |

### Pasos a seguir

1. **Definir el área de estudio** del glaciar (celda de partida más abajo, reutiliza las coordenadas de tu tarea de K-Means).
2. **Cargar y visualizar** un composito Sentinel-2 de época seca.
3. **Recolectar puntos de entrenamiento** (clic interactivo o predefinidos) para las 4 clases — recuerda **verificar visualmente** cada punto antes de entrenar.
4. **Extraer valores espectrales y normalizar** (igual que hicimos para la Laguna Junín).
5. **Entrenar al menos dos clasificadores SVM**: uno lineal y uno RBF (con y sin normalizar, para comparar igual que en la sección 7).
6. **Clasificar la imagen completa** y visualizar en el mapa con leyenda.
7. **Comparar con tu resultado de K-Means** (tarea anterior): ¿coinciden las clases? ¿Cuál te parece más confiable y por qué?

> **Pista clave:** la nieve/hielo es muy brillante en el visible y muy oscura en SWIR — un contraste fuerte que debería facilitar la separación con SVM lineal incluso sin kernel RBF.

In [110]:
# ── Configuración: área de estudio del glaciar (misma que en la tarea de K-Means) ──
import ee, geemap
ee.Authenticate()
ee.Initialize(project='utec-ingambiental-01')  # <-- REEMPLAZA con tu Project ID ee-cmillanarancibia

# Glaciar Coropuna (Arequipa)
LON_MIN_G, LAT_MIN_G = -72.766883, -15.417441
LON_MAX_G, LAT_MAX_G = -72.508209, -15.65

area_glaciar = ee.Geometry.Rectangle([LON_MIN_G, LAT_MIN_G, LON_MAX_G, LAT_MAX_G])

composito_g = (
    ee.ImageCollection('NASA/HLS/HLSL30/v002')
    .filterBounds(area_glaciar)
    .filterDate('2025-05-01', '2025-09-30')
    #.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .select(['B2','B3','B4','B5','B6','B7'])
    .median()
    .clip(area_glaciar)
)
composito_g = composito_g.select(
    ['B2','B3','B4','B5','B6','B7'], ['Azul','Verde','Rojo','NIR','SWIR1','SWIR2']
)

Map_g = geemap.Map(basemap="Esri.WorldTopoMap")
Map_g.centerObject(area_glaciar, zoom=13)
Map_g.addLayer(composito_g, {'bands':['Rojo','Verde','Azul'],'min':0,'max':0.6}, 'Sentinel-2 — Glaciar')
Map_g

Map(center=[-15.533736301309274, -72.63754599999886], controls=(WidgetControl(options=['position', 'transparen…

### Tu turno

Completa las celdas siguientes siguiendo el mismo patrón usado para la Laguna Junín (sección 7).

In [99]:
import ipywidgets as widgets
from IPython.display import display

# ── Definiciones de clases ──
class_labels = {
    'Agua': 0,
    'Nieve/Hielo': 1,
    'Roca/Morrena': 2,
    'Pastizal ralo': 3
}

# Almacenamiento de puntos recopilados: {class_code: [(lon, lat), ...]}
collected_points = {v: [] for v in class_labels.values()}

# ── Menú desplegable para seleccionar la clase actual ──
class_dropdown = widgets.Dropdown(
    options=list(class_labels.keys()),
    value='Agua',
    description='Clase:',
    style={'description_width': 'initial'}
)

# ── Registro de salida ──
log_output = widgets.Output()

# ── Controlador de clics ──
def handle_map_click(**kwargs):
    if kwargs.get('type') == 'click':
        coords = kwargs.get('coordinates') # [lat, lon]
        lat, lon = coords[0], coords[1]
        current_class = class_labels[class_dropdown.value]
        collected_points[current_class].append((lon, lat))
        # Agregar un marcador al mapa
        from ipyleaflet import Marker, AwesomeIcon
        # Se corrigió 'cyan' a 'lightblue' que es soportado por AwesomeIcon
        colors = { 0: 'blue', 1: 'lightblue', 2: 'orange', 3: 'green' }
        icon = AwesomeIcon(name='map-marker', marker_color=colors[current_class])

        marker = Marker(location=(lat, lon), icon=icon, draggable=False)

        Map_g.add(marker)

        with log_output:
            total = sum(len(v) for v in collected_points.values())
            print(
                f"[{class_dropdown.value}] "
                f"Punto añadido ({lon:.4f}, {lat:.4f}) | "
                f"Total: {total} puntos"
            )

# Registrar el controlador de clic
Map_g.on_interaction(handle_map_click)

# ── Mostrar controles ──
botón_borrar = widgets.Button(
    description='Borrar todos los puntos',
    button_style='warning'
)

def on_clear(b):
    for k in collected_points:
        collected_points[k] = []

    with log_output:
        log_output.clear_output()
        print("Todos los puntos borrados. (Los marcadores permanecen en el mapa durante referencia.)")

botón_borrar.on_click(on_clear)

print("Haz clic en el mapa para obtener puntos de entrenamiento.")
print("Selecciona una clase del menú desplegable y luego haz clic en las ubicaciones del mapa.\n")

display(widgets.HBox([class_dropdown, botón_borrar]))
display(log_output)

Haz clic en el mapa para obtener puntos de entrenamiento.
Selecciona una clase del menú desplegable y luego haz clic en las ubicaciones del mapa.



Output()

In [100]:
# ── Convertir los puntos clicados en un ee.FeatureCollection ──

class_names = {
    0: 'Agua',
    1: 'Nieve/Hielo',
    2: 'Roca/Morrena',
    3: 'Pastizal ralo'
}

features = []

for class_code, points in collected_points.items():
    for lon, lat in points:
        feat = ee.Feature(
            ee.Geometry.Point([lon, lat]),
            {'landcover': class_code}
        )
        features.append(feat)

puntos = ee.FeatureCollection(features)

# ── Summary ──
print("Training points collected:")

for class_code, points in collected_points.items():
    print(f"  {class_names[class_code]:15s}: {len(points)} points")

print(f"  {'TOTAL':15s}: {puntos.size().getInfo()} points")

Training points collected:
  Agua           : 14 points
  Nieve/Hielo    : 10 points
  Roca/Morrena   : 22 points
  Pastizal ralo  : 10 points
  TOTAL          : 56 points


In [101]:
# TU CÓDIGO AQUÍ — Paso 3: recolectar puntos de entrenamiento para las 4 clases del glaciar
# Pista: reutiliza el widget interactivo de la sección 7, o define puntos predefinidos
#        con ee.Feature(ee.Geometry.Point([lon, lat]), {'landcover': codigo})
agua = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([-72.6460, -15.5120]), {'landcover': 0}),
    ee.Feature(ee.Geometry.Point([-72.6460, -15.5976]), {'landcover': 0}),
])

nieve = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([-72.6371, -15.5044]), {'landcover': 1}),
    ee.Feature(ee.Geometry.Point([-72.6371, -15.5679]), {'landcover': 1}),
])

roca = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([-72.6773, -15.5624]), {'landcover': 2}),
    ee.Feature(ee.Geometry.Point([-72.6773, -15.6021]), {'landcover': 2}),
])

pastizal = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([-72.6833, -15.5810]), {'landcover': 3}),
])

puntos = agua.merge(nieve).merge(roca).merge(pastizal)

print(f"Total puntos de entrenamiento: {puntos.size().getInfo()}")

Total puntos de entrenamiento: 7


In [105]:
# TU CÓDIGO AQUÍ — Paso 4: extraer valores espectrales y normalizar (z-score)
# Pista: reutiliza sampleRegions + reduceRegion(mean/stdDev), igual que con la Laguna Junín
bandas = ['Azul','Verde','Rojo','NIR','SWIR1','SWIR2']
datos_entrenamiento = composito_g.select(bandas).sampleRegions(collection=puntos,properties=['landcover'],scale=30).filter(ee.Filter.notNull(bandas))
means = composito_g.select(bandas).reduceRegion( reducer=ee.Reducer.mean(), geometry=area_glaciar, scale=30, maxPixels=1e9)
stds = composito_g.select(bandas).reduceRegion(  reducer=ee.Reducer.stdDev(), geometry=area_glaciar, scale=30, maxPixels=1e9)
bandas_norm = [b+'_norm' for b in bandas]
composito_norm = ee.Image.cat([
    composito_g.select(b).subtract(ee.Number(means.get(b))).divide(ee.Number(stds.get(b))).rename(b+'_norm')
    for b in bandas
])
datos_entrenamiento_norm = composito_norm.sampleRegions( collection=puntos,properties=['landcover'],scale=30).filter(ee.Filter.notNull(bandas_norm))
print("Se crearon bandas normalizadas y se remuestrearon los datos de entrenamiento.")


Se crearon bandas normalizadas y se remuestrearon los datos de entrenamiento.


In [106]:
# TU CÓDIGO AQUÍ — Paso 5: entrenar SVM lineal y RBF (con y sin normalizar)
# Pista: ee.Classifier.libsvm(kernelType=..., cost=..., gamma=...)
svm_lineal = ee.Classifier.libsvm(
    kernelType='LINEAR', cost=10
).train(datos_entrenamiento, classProperty='landcover', inputProperties=bandas)

svm_rbf = ee.Classifier.libsvm(
    kernelType='RBF', cost=10, gamma=0.5
).train(datos_entrenamiento, classProperty='landcover', inputProperties=bandas)

svm_rbf_norm = ee.Classifier.libsvm(
    kernelType='RBF', cost=10, gamma=0.01
).train(datos_entrenamiento_norm, classProperty='landcover', inputProperties=bandas_norm)

print("Tres clasificadores SVM entrenados:")
print(" 1. Lineal          — bandas crudas")
print(" 2. RBF sin escalar — bandas crudas, gamma=0.5")
print(" 3. RBF normalizado — z-score,       gamma=0.01")

Tres clasificadores SVM entrenados.


In [108]:
# TU CÓDIGO AQUÍ — Paso 6: clasificar la imagen completa y visualizar con leyenda
# Pista: composito_g.classify(modelo) + geemap.Map().addLayer(...) + Map.add_legend(...)

clasi_lineal   = composito_g.select(bandas).classify(svm_lineal)
clasi_rbf      = composito_g.select(bandas).classify(svm_rbf)
clasi_rbf_norm = composito_norm.classify(svm_rbf_norm)

paleta = ['#1f78b4','#00e5ff','#8c6d46','#7ec841']  # Agua, Nieve/Hielo, Roca/Morrena, Pastizal ralo

Map2 = geemap.Map(basemap="Esri.WorldTopoMap")
Map2.centerObject(area_glaciar, zoom=12)
Map2.addLayer(composito_g,{'bands':['Rojo','Verde','Azul'],'min':0,'max':0.6}, 'HLS base — Glaciar Coropuna')
Map2.addLayer( clasi_lineal, {'min':0,'max':3,'palette':paleta}, 'SVM Lineal')
Map2.addLayer( clasi_rbf, {'min':0,'max':3,'palette':paleta},'SVM RBF (sin normalizar)')
Map2.addLayer(clasi_rbf_norm,{'min':0,'max':3,'palette':paleta},'SVM RBF (normalizado)')
Map2.add_legend(title='Cobertura',keys=['Agua','Nieve/Hielo','Roca/Morrena','Pastizal ralo'],colors=paleta)

Map2


Map(center=[-15.533736301309277, -72.63754600000026], controls=(WidgetControl(options=['position', 'transparen…

### Para tu informe

Responde brevemente (puedes usar esta misma celda en formato Markdown):


1. ¿Qué kernel (lineal o RBF) dio un mapa más coherente para el glaciar? ¿Por qué crees que ocurrió eso?


El lineal. En su mapa el casquete del Coropuna sale bien delimitado y no aparece agua dentro del hielo. En los dos RBF sí pasa: parte del hielo, sobre todo en sombra, se clasificó como agua. Creo que se debe a que la nieve se distingue mucho del resto, porque es muy brillante en el visible y muy oscura en SWIR, así que una frontera recta ya la separa. El RBF es más flexible, y con tan pocos puntos de entrenamiento se ajustó demasiado a los ejemplos que tenía, incluso a los malos.


2. Compara el mapa de SVM con el de K-Means (tarea anterior, mismo glaciar). ¿Las clases de "nieve/hielo" coinciden espacialmente?

En general sí, los dos métodos identifican el casquete del Coropuna en la misma zona. Sin embargo, hay algunas diferencias en los bordes y en las zonas de sombra. En el SVM la nieve/hielo se ve más definida porque las clases fueron seleccionadas previamente mediante puntos de entrenamiento, mientras que en K-Means los grupos fueron formados automáticamente

3. ¿Qué ventaja tiene SVM (supervisado) sobre K-Means (no supervisado) para este problema? ¿Y qué desventaja?

La ventaja del SVM es que puedo indicarle al modelo qué zonas corresponden a agua, nieve, roca o pastizal, por lo que tengo más control sobre la clasificación. La desventaja es que depende bastante de los puntos que seleccione. Si pongo pocos puntos o alguno está mal ubicado, el modelo puede aprender mal y clasificar otras zonas de forma incorrecta. En cambio, K-Means no necesita puntos de entrenamiento, aunque después hay que interpretar qué representa cada grupo.
